In [9]:
# =====================================================================
# MULTISOURCE PATCH EXTRACTION FOR DEEP LEARNING
# =====================================================================

import numpy as np
import pandas as pd
import geopandas as gpd
import os
import time
import json
from pathlib import Path
from scipy.spatial import cKDTree
import rasterio
from rasterio.transform import from_bounds
from rasterio.crs import CRS
from shapely.geometry import box, Point
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from rasterio.transform import rowcol
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [10]:
# =====================================================================
# CONFIGURATION
# =====================================================================

ROOT_DIRECTORY = 'D:\\Tese_data_refinada'
ROOT_DIRECTORY_DATA = os.path.join(ROOT_DIRECTORY, 'DATA')
OUTPUT_BASE_DIR = os.path.join(ROOT_DIRECTORY, '_multisource_patches')
VALIDATION_DIR = os.path.join(ROOT_DIRECTORY, '_validation_qgis')
EXAMPLES_DIR = os.path.join(ROOT_DIRECTORY, '_example_patches')

os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)
os.makedirs(VALIDATION_DIR, exist_ok=True)
os.makedirs(EXAMPLES_DIR, exist_ok=True)

PATCH_SIZE = 15  
PIXEL_SIZE = 25  
BIOMASS_POOL = 'HEIGHT'

print(f"Root directory: {ROOT_DIRECTORY}")
print(f"Output directory: {OUTPUT_BASE_DIR}")
print(f"Patch configuration: {PATCH_SIZE}x{PATCH_SIZE}, {PIXEL_SIZE}m per pixel")

Root directory: D:\Tese_data_refinada
Output directory: D:\Tese_data_refinada\_multisource_patches
Patch configuration: 15x15, 25m per pixel


In [11]:
# =====================================================================
# DISCOVER AND LOAD DATA
# =====================================================================

def discover_data_structure(root_dir):

    data_structure = {}
    
    for directory in os.listdir(root_dir):
        dir_path = os.path.join(root_dir, directory)
        
        if not os.path.isdir(dir_path):
            continue
        
        area_name = directory.split('_')[0]
        
        if area_name not in data_structure:
            data_structure[area_name] = {
                'gpkg': None,
                'rasters': [],
                'crs': None
            }
        
        # Find GeoPackage (reference points)
        for file in os.listdir(dir_path):
            if file.endswith('.gpkg'):
                gpkg_path = os.path.join(dir_path, file)
                data_structure[area_name]['gpkg'] = gpkg_path
                
                # Load and get CRS
                gdf = gpd.read_file(gpkg_path)
                data_structure[area_name]['crs'] = gdf.crs
                print(f"Found GeoPackage for {area_name}: {file}")
                print(f"  CRS: {gdf.crs}")
                print(f"  Points: {len(gdf)}")
            
            # Find rasters (satellite data)
            elif file.endswith('.tif'):
                raster_path = os.path.join(dir_path, file)
                data_structure[area_name]['rasters'].append(raster_path)
        
        if data_structure[area_name]['rasters']:
            print(f"  Found {len(data_structure[area_name]['rasters'])} rasters")
    
    return data_structure


# Discover data
print("Discovering data structure...")
data_structure = discover_data_structure(ROOT_DIRECTORY_DATA)

print(f"\nDiscovered {len(data_structure)} areas:")
for area, info in data_structure.items():
    print(f"  {area}:")
    print(f"    - GeoPackage: {os.path.basename(info['gpkg']) if info['gpkg'] else 'NOT FOUND'}")
    print(f"    - Rasters: {len(info['rasters'])}")
    print(f"    - CRS: {info['crs']}")

# Filter valid areas (those with both gpkg and rasters)
valid_areas = {k: v for k, v in data_structure.items() 
               if v['gpkg'] is not None and len(v['rasters']) > 0}

print(f"\nValid areas for processing: {list(valid_areas.keys())}")

Discovering data structure...
Found GeoPackage for Mafra: Mafra_20200000_GEDI_AGBD_ALL.gpkg
  CRS: EPSG:3763
  Points: 463
  Found 306 rasters
Found GeoPackage for Monsanto: Monsanto_20200000_GEDI_AGBD_ALL.gpkg
  CRS: EPSG:3763
  Points: 455
  Found 306 rasters
Found GeoPackage for Pombal: Pombal_20200000_GEDI_AGBD_ALL.gpkg
  CRS: EPSG:3763
  Points: 850
  Found 306 rasters
Found GeoPackage for SerraLousa: SerraLousa_20200000_GEDI_AGBD_ALL.gpkg
  CRS: EPSG:3763
  Points: 5825
  Found 306 rasters
Found GeoPackage for SintraCascais: SintraCascais_20200000_GEDI_AGBD_ALL.gpkg
  CRS: EPSG:3763
  Points: 736
  Found 306 rasters
Found GeoPackage for VPA: VPA_20200000_GEDI_AGBD_ALL.gpkg
  CRS: EPSG:3763
  Points: 535
  Found 306 rasters

Discovered 6 areas:
  Mafra:
    - GeoPackage: Mafra_20200000_GEDI_AGBD_ALL.gpkg
    - Rasters: 306
    - CRS: EPSG:3763
  Monsanto:
    - GeoPackage: Monsanto_20200000_GEDI_AGBD_ALL.gpkg
    - Rasters: 306
    - CRS: EPSG:3763
  Pombal:
    - GeoPackage: Pomb

In [12]:
# =====================================================================
# INTERPOLATION FUNCTIONS
# =====================================================================

def cubic_kernel(t):
    """
    Cubic interpolation kernel (Catmull-Rom spline).
    """
    t = np.abs(t)
    a = -0.5
    
    return np.where(
        t <= 1,
        (a + 2) * t**3 - (a + 3) * t**2 + 1,
        np.where(
            t < 2,
            a * t**3 - 5 * a * t**2 + 8 * a * t - 4 * a,
            0
        )
    )


def bicubic_interpolation(image, x, y, no_data_value):
    """
    Bicubic interpolation handling NaN values.
    
    Returns: interpolated_value (float), success (bool)
    """
    height, width = image.shape
    
    # Handle border cases
    if x < 1 or x > width - 3 or y < 1 or y > height - 3:
        return no_data_value, False
    
    # Floor values
    x0 = int(np.floor(x))
    y0 = int(np.floor(y))
    
    # Compute distances
    dx = x - x0
    dy = y - y0
    
    # Extract 4x4 neighborhood
    neighborhood = image[y0-1:y0+3, x0-1:x0+3].astype(np.float32)
    
    # Mask valid (non-nodata) values
    valid_mask = ~np.isin(neighborhood, no_data_value)
    
    if valid_mask.sum() < 8:  # Not enough valid neighbors
        return no_data_value, False
    
    # Replace nodata with mean of valid values
    valid_vals = neighborhood[valid_mask]
    if len(valid_vals) > 0:
        neighborhood[neighborhood == no_data_value] = np.mean(valid_vals)
    
    # Compute weights
    wx = cubic_kernel(np.arange(-1, 3) - dx)
    wy = cubic_kernel(np.arange(-1, 3) - dy)
    
    # Apply valid mask to weights
    wx_masked = wx * valid_mask.sum(axis=0)
    wy_masked = wy * valid_mask.sum(axis=1)
    
    # Normalize weights
    if np.sum(wx_masked) > 0:
        wx_masked /= np.sum(wx_masked)
    if np.sum(wy_masked) > 0:
        wy_masked /= np.sum(wy_masked)
    
    # Interpolate
    intermediate = np.dot(neighborhood, wx_masked)
    result = np.dot(intermediate, wy_masked)
    
    # Verify result is within valid range
    valid_range_min = np.min(valid_vals)
    valid_range_max = np.max(valid_vals)
    
    if not (valid_range_min <= result <= valid_range_max):
        return no_data_value, False
    
    return result, True


def bilinear_interpolation(image, x, y, no_data_value):
    """
    Bilinear interpolation handling NaN values.
    
    Returns: interpolated_value (float), success (bool)
    """
    height, width = image.shape
    
    # Get integer grid points
    x0, y0 = int(np.floor(x)), int(np.floor(y))
    x1 = min(x0 + 1, width - 1)
    y1 = min(y0 + 1, height - 1)
    
    # Extract 2x2 neighborhood
    neighborhood = np.array([
        [image[y0, x0], image[y0, x1]],
        [image[y1, x0], image[y1, x1]]
    ], dtype=np.float32)
    
    # Valid mask
    valid_mask = ~np.isin(neighborhood, no_data_value)
    
    if valid_mask.sum() < 2:
        return no_data_value, False
    
    # Replace nodata with mean
    valid_vals = neighborhood[valid_mask]
    if len(valid_vals) > 0:
        neighborhood[~valid_mask] = np.mean(valid_vals)
    
    # Compute weights
    wx = x - x0
    wy = y - y0
    
    # Interpolate along x
    weights_x = np.array([1 - wx, wx])
    sum_weights_x = np.dot(weights_x, valid_mask)
    
    fx_y = np.dot(weights_x, neighborhood)
    fx_y = np.where(sum_weights_x > 0, fx_y / sum_weights_x, np.nan)
    
    # Interpolate along y
    weights_y = np.array([1 - wy, wy])
    valid_y = ~np.isnan(fx_y)
    
    if not valid_y.any():
        return no_data_value, False
    
    f_xy = np.dot(weights_y, np.nan_to_num(fx_y))
    f_xy /= np.dot(weights_y, valid_y.astype(float))
    
    # Verify range
    if not (np.min(valid_vals) <= f_xy <= np.max(valid_vals)):
        return no_data_value, False
    
    return f_xy, True


def get_raster_value_at_coord(raster_data, no_data, transform, height, width, coord_x, coord_y, use_bicubic=True):
    """
    Extract interpolated value from raster at geographic coordinate.
    
    Returns: (value, interpolation_method_used)
    """
    
    # CORREÇÃO: Obter coordenadas em float (com casas decimais) para permitir interpolação
    col, row = ~transform * (coord_x, coord_y)
    
    # Para ler o array e validar limites, usamos os valores inteiros (Nearest)
    int_row, int_col = int(np.floor(row)), int(np.floor(col))
        
    # Check bounds
    if int_row < 0 or int_col < 0 or int_row >= height or int_col >= width:
        return np.nan, 'out_of_bounds'
        
    # Try to read nearest neighbor value
    nn_value = raster_data[int_row, int_col]
    
    if nn_value == no_data or np.isnan(nn_value):
        return np.nan, 'nodata'
    
    # Try bicubic (NOTA: passamos 'col' e 'row' que têm as casas decimais!)
    if use_bicubic and int_row > 0 and int_col > 0 and \
       int_row < height - 2 and int_col < width - 2:
        value, success = bicubic_interpolation(raster_data, col, row, no_data)
        if success and not np.isnan(value):
            return value, 'bicubic'
    
    # Try bilinear
    if int_row < height - 1 and int_col < width - 1:
        value, success = bilinear_interpolation(raster_data, col, row, no_data)
        if success and not np.isnan(value):
            return value, 'bilinear'
    
    # Return nearest neighbor fallback
    return nn_value, 'nearest'


In [13]:
# =====================================================================
# PATCH GENERATION UTILITIES
# =====================================================================

def generate_patch_coordinates(center_x, center_y, patch_size=15, pixel_size=25):
    """
    Generate geographic coordinates for a 15x15 patch grid.
    
    The patch center corresponds exactly to (center_x, center_y).
    Each cell is separated by pixel_size (25m).
    
    Returns: coords array of shape (patch_size, patch_size, 2)
             where coords[i, j] = [x, y] geographic coordinate
    """
    
    half_patch = (patch_size - 1) / 2.0
    offset = half_patch * pixel_size
    
    coords = np.zeros((patch_size, patch_size, 2), dtype=np.float64)
    
    for i in range(patch_size):
        for j in range(patch_size):
            # i is row (y direction), j is column (x direction)
            x = center_x - offset + j * pixel_size
            y = center_y + offset - i * pixel_size  # top is +y
            coords[i, j, 0] = x
            coords[i, j, 1] = y
    
    return coords


def get_patch_bounds(center_x, center_y, patch_size=15, pixel_size=25):
    """
    Get bounding box (minx, miny, maxx, maxy) of patch.
    """
    half_patch = (patch_size - 1) / 2.0
    offset = half_patch * pixel_size
    
    minx = center_x - offset
    maxx = center_x + offset
    miny = center_y - offset
    maxy = center_y + offset
    
    return (minx, miny, maxx, maxy)


def create_patch_polygon(center_x, center_y, patch_size=15, pixel_size=25):
    """
    Create Shapely polygon for patch extent (for QGIS).
    """
    minx, miny, maxx, maxy = get_patch_bounds(center_x, center_y, patch_size, pixel_size)
    return box(minx, miny, maxx, maxy)

print("Patch utilities loaded successfully!")

Patch utilities loaded successfully!


In [14]:
# =====================================================================
# MAIN PROCESSING PIPELINE (SUPER OTIMIZADO COM JOBLIB)
# =====================================================================

import time
from joblib import Parallel, delayed

print("\n" + "="*70)
print("STARTING MULTIPROCESSING PATCH EXTRACTION")
print("="*70)

all_results = {}

# 1. Função "Worker" que vai correr em paralelo para cada Raster
def process_single_raster(c_idx, r_path, r_name, N, patch_size, precomputed_coords, no_data_val=None):
    """
    Lê um único raster e extrai os patches para todos os pontos.
    Isto correrá em paralelo noutros núcleos do CPU.
    """
    X_raster = np.zeros((N, patch_size, patch_size), dtype=np.float32)
    nan_counts = np.zeros(N, dtype=int)
    
    # Metadados locais para este raster
    meta_raster = {
        n_idx: {'bicubic': 0, 'bilinear': 0, 'nearest': 0, 'nodata': 0, 'out_of_bounds': 0, 'total': patch_size**2}
        for n_idx in range(N)
    }
    
    with rasterio.open(r_path) as src:
        raster_data = src.read(1)
        no_data = src.nodata if src.nodata is not None else no_data_val
        transform = src.transform
        h, w = src.height, src.width
        
        for n_idx in range(N):
            patch_coords = precomputed_coords[n_idx]
            patch = np.zeros((patch_size, patch_size), dtype=np.float32)
            nan_cnt = 0
            
            for i in range(patch_size):
                for j in range(patch_size):
                    px, py = patch_coords[i, j]
                    val, method = get_raster_value_at_coord(raster_data, no_data, transform, h, w, px, py)
                    patch[i, j] = val
                    
                    meta_raster[n_idx][method] += 1
                    
                    if np.isnan(val):
                        nan_cnt += 1
            
            X_raster[n_idx, :, :] = patch
            nan_counts[n_idx] = nan_cnt
            
    return c_idx, r_name, X_raster, nan_counts, meta_raster


# 2. O Loop Principal pelas Áreas
for area_name, area_info in valid_areas.items():
    print(f"\nProcessing area: {area_name}")
    area_start_time = time.perf_counter()
    
    gdf_points = gpd.read_file(area_info['gpkg'])
    raster_paths = sorted(area_info['rasters'])
    
    N = len(gdf_points)
    C = len(raster_paths)
    raster_names = [Path(p).stem for p in raster_paths]
    
    # Pré-calcular coordenadas para TODOS os pontos desta área
    print("  Pre-computing patch coordinates...")
    precomputed_coords = []
    for _, row in gdf_points.iterrows():
        precomputed_coords.append(generate_patch_coordinates(row.geometry.x, row.geometry.y, PATCH_SIZE, PIXEL_SIZE))
    
    # Inicializar tensores principais
    X = np.zeros((N, C, PATCH_SIZE, PATCH_SIZE), dtype=np.float32)
    y = gdf_points[BIOMASS_POOL].values.astype(np.float32) if BIOMASS_POOL in gdf_points.columns else np.zeros(N)
    is_valid_sample = np.ones(N, dtype=bool)
    
    # Metadados principais
    sample_metadata = {n_idx: {} for n_idx in range(N)}

    print(f"  Extracting patches from {C} rasters using Parallel Processing...")
    
    # =================================================================
    # A MAGIA ACONTECE AQUI: n_jobs=-1 usa todos os núcleos do teu CPU
    # =================================================================
    results = Parallel(n_jobs=-1, backend="loky")(
        delayed(process_single_raster)(
            c_idx, r_path, raster_names[c_idx], N, PATCH_SIZE, precomputed_coords
        ) for c_idx, r_path in enumerate(raster_paths)
    )
    
    # Reconstruir o tensor X com os resultados vindos do multiprocessamento
    for res in results:
        c_idx, r_name, X_raster, nan_counts, meta_raster = res
        
        # Guardar a matriz no grande tensor X
        X[:, c_idx, :, :] = X_raster
        
        for n_idx in range(N):
            sample_metadata[n_idx][r_name] = meta_raster[n_idx]
            # Se tiver mais de 50% de NaNs, invalida a amostra
            if nan_counts[n_idx] / (PATCH_SIZE**2) > 0.5:
                is_valid_sample[n_idx] = False

    # Organizar amostras finais
    valid_indices = np.where(is_valid_sample)[0]
    X_final = X[valid_indices]
    y_final = y[valid_indices]
    
    formatted_samples = []
    for n_idx in range(N):
        sample_patches = {raster_names[c_idx]: X[n_idx, c_idx, :, :] for c_idx in range(C)}
        formatted_samples.append({
            'point_id': n_idx,
            'center': (gdf_points.iloc[n_idx].geometry.x, gdf_points.iloc[n_idx].geometry.y),
            'target': y[n_idx],
            'patches': sample_patches,
            'valid': is_valid_sample[n_idx],
            'raster_names': raster_names,
            'metadata': sample_metadata[n_idx]
        })

    all_results[area_name] = {
        'X': X_final, 'y': y_final, 'raster_names': raster_names,
        'valid_indices': valid_indices.tolist(), 'samples': formatted_samples,
        'gdf_points': gdf_points, 'patch_size': PATCH_SIZE,
        'pixel_size': PIXEL_SIZE, 'crs': gdf_points.crs
    }
    
    elapsed = time.perf_counter() - area_start_time
    print(f"Finished {area_name} in {elapsed/60:.2f} minutes.")

print("\n" + "="*70)
print("EXTRACTION COMPLETE")
print("="*70)


STARTING MULTIPROCESSING PATCH EXTRACTION

Processing area: Mafra
  Pre-computing patch coordinates...
  Extracting patches from 306 rasters using Parallel Processing...
Finished Mafra in 13.70 minutes.

Processing area: Monsanto
  Pre-computing patch coordinates...
  Extracting patches from 306 rasters using Parallel Processing...
Finished Monsanto in 13.60 minutes.

Processing area: Pombal
  Pre-computing patch coordinates...
  Extracting patches from 306 rasters using Parallel Processing...
Finished Pombal in 17.98 minutes.

Processing area: SerraLousa
  Pre-computing patch coordinates...
  Extracting patches from 306 rasters using Parallel Processing...
Finished SerraLousa in 95.08 minutes.

Processing area: SintraCascais
  Pre-computing patch coordinates...
  Extracting patches from 306 rasters using Parallel Processing...
Finished SintraCascais in 17.24 minutes.

Processing area: VPA
  Pre-computing patch coordinates...
  Extracting patches from 306 rasters using Parallel Proces

In [21]:
# =====================================================================
# SAVE TENSORS FOR DEEP LEARNING
# =====================================================================

print("\nSaving tensors to disk...")

for area_name, result in all_results.items():
    
    # Create area output directory
    area_output_dir = os.path.join(OUTPUT_BASE_DIR, area_name)
    os.makedirs(area_output_dir, exist_ok=True)
    
    # Save tensors
    np.save(os.path.join(area_output_dir, 'X.npy'), result['X'])
    np.save(os.path.join(area_output_dir, 'y.npy'), result['y'])
    
    # Save metadata
    metadata = {
        'patch_size': result['patch_size'],
        'pixel_size': result['pixel_size'],
        'num_channels': len(result['raster_names']),
        'num_samples': result['X'].shape[0],
        'raster_names': result['raster_names'],
        'valid_sample_indices': result['valid_indices'],
        'crs': str(result['crs']),
        'nan_percentage': float(np.isnan(result['X']).sum() / result['X'].size * 100)
    }
    
    with open(os.path.join(area_output_dir, 'metadata.json'), 'w') as f:
        json.dump(metadata, f, indent=2)
    
    # Print summary
    print(f"\n{area_name}:")
    print(f"  X: {result['X'].shape} ({result['X'].nbytes / 1e9:.2f} GB)")
    print(f"  y: {result['y'].shape} ({result['y'].nbytes / 1e6:.2f} MB)")
    print(f"  Saved to: {area_output_dir}")

print("\n✓ Tensors saved successfully!")


Saving tensors to disk...

Mafra:
  X: (389, 306, 15, 15) (0.11 GB)
  y: (389,) (0.00 MB)
  Saved to: D:\Tese_data_refinada\_multisource_patches\Mafra

Monsanto:
  X: (418, 306, 15, 15) (0.12 GB)
  y: (418,) (0.00 MB)
  Saved to: D:\Tese_data_refinada\_multisource_patches\Monsanto

Pombal:
  X: (792, 306, 15, 15) (0.22 GB)
  y: (792,) (0.01 MB)
  Saved to: D:\Tese_data_refinada\_multisource_patches\Pombal

SerraLousa:
  X: (5299, 306, 15, 15) (1.46 GB)
  y: (5299,) (0.04 MB)
  Saved to: D:\Tese_data_refinada\_multisource_patches\SerraLousa

SintraCascais:
  X: (602, 306, 15, 15) (0.17 GB)
  y: (602,) (0.00 MB)
  Saved to: D:\Tese_data_refinada\_multisource_patches\SintraCascais

VPA:
  X: (509, 306, 15, 15) (0.14 GB)
  y: (509,) (0.00 MB)
  Saved to: D:\Tese_data_refinada\_multisource_patches\VPA

✓ Tensors saved successfully!


In [22]:
# =====================================================================
# GENERATE QGIS VALIDATION OUTPUT
# =====================================================================

print("\nGenerating QGIS validation GeoPackages...")

for area_name, result in all_results.items():
    
    samples = result['samples']
    gdf_points = result['gdf_points']
    
    # Create geometries for patch extents
    patch_geometries = []
    patch_attributes = {
        'point_id': [],
        'target': [],
        'valid': [],
        'center_x': [],
        'center_y': [],
        'nan_count': [],
        'nan_percent': []
    }
    
    for sample in samples:
        point_id = sample['point_id']
        center_x, center_y = sample['center']
        
        # Create patch polygon
        patch_poly = create_patch_polygon(
            center_x, center_y, 
            PATCH_SIZE, PIXEL_SIZE
        )
        patch_geometries.append(patch_poly)
        
        # Collect attributes
        patch_attributes['point_id'].append(point_id)
        patch_attributes['target'].append(sample['target'])
        patch_attributes['valid'].append(sample['valid'])
        patch_attributes['center_x'].append(center_x)
        patch_attributes['center_y'].append(center_y)
        
        # Count NaN values in first raster patch
        if sample['patches']:
            first_patch = list(sample['patches'].values())[0]
            nan_cnt = int(np.isnan(first_patch).sum())
            nan_pct = 100 * nan_cnt / first_patch.size
        else:
            nan_cnt = -1
            nan_pct = -1.0
        
        patch_attributes['nan_count'].append(nan_cnt)
        patch_attributes['nan_percent'].append(nan_pct)
    
    # Create GeoDataFrame
    gdf_patches = gpd.GeoDataFrame(
        patch_attributes,
        geometry=patch_geometries,
        crs=result['crs']
    )
    
    # Save to GeoPackage
    output_path = os.path.join(
        VALIDATION_DIR, 
        f'{area_name}_patch_extents.gpkg'
    )
    gdf_patches.to_file(output_path, driver='GPKG')
    
    print(f"  {area_name}: {output_path}")
    print(f"    Valid: {gdf_patches['valid'].sum()}/{len(gdf_patches)}")

print("\n✓ QGIS validation files ready!")


Generating QGIS validation GeoPackages...
  Mafra: D:\Tese_data_refinada\_validation_qgis\Mafra_patch_extents.gpkg
    Valid: 389/463
  Monsanto: D:\Tese_data_refinada\_validation_qgis\Monsanto_patch_extents.gpkg
    Valid: 418/455
  Pombal: D:\Tese_data_refinada\_validation_qgis\Pombal_patch_extents.gpkg
    Valid: 792/850
  SerraLousa: D:\Tese_data_refinada\_validation_qgis\SerraLousa_patch_extents.gpkg
    Valid: 5299/5825
  SintraCascais: D:\Tese_data_refinada\_validation_qgis\SintraCascais_patch_extents.gpkg
    Valid: 602/736
  VPA: D:\Tese_data_refinada\_validation_qgis\VPA_patch_extents.gpkg
    Valid: 509/535

✓ QGIS validation files ready!


In [23]:
# segunda versao com multibandas
# =====================================================================
# EXPORT EXAMPLE PATCHES AS GEOREFERENCED MULTIBAND GEOTIFFS
# =====================================================================

print("\nExporting example multisensor patches as GeoTIFFs...")

def export_multiband_patch_geotiff(patches_dict, bounds, output_path, crs):
    """
    Export a multi-sensor patch as a georeferenced multiband GeoTIFF.
    """
    minx, miny, maxx, maxy = bounds

    raster_names = list(patches_dict.keys())
    first_patch = patches_dict[raster_names[0]]

    height, width = first_patch.shape
    count = len(raster_names)

    transform = from_bounds(minx, miny, maxx, maxy, width, height)

    with rasterio.open(
        output_path, 'w',
        driver='GTiff',
        height=height,
        width=width,
        count=count,
        dtype=rasterio.float32,
        crs=crs,
        transform=transform,
        nodata=np.nan
    ) as dst:
        for band_idx, raster_name in enumerate(raster_names, start=1):
            patch_data = patches_dict[raster_name].astype(np.float32)
            dst.write(patch_data, band_idx)
            dst.set_band_description(band_idx, raster_name)

for area_name, result in all_results.items():

    samples = result['samples']
    crs = result['crs']

    area_examples_dir = os.path.join(EXAMPLES_DIR, area_name)
    os.makedirs(area_examples_dir, exist_ok=True)

    example_count = 0
    for sample in samples:

        point_id = sample['point_id']
        center_x, center_y = sample['center']
        bounds = get_patch_bounds(center_x, center_y, PATCH_SIZE, PIXEL_SIZE)

        output_file = os.path.join(
            area_examples_dir,
            f'point_{point_id:04d}_multisensor_patch.tif'
        )

        export_multiband_patch_geotiff(sample['patches'], bounds, output_file, crs)

        example_count += 1
        print(f"  {area_name}: example {example_count} (point {point_id})")

print("\n✓ Example multiband GeoTIFFs exported!")


Exporting example multisensor patches as GeoTIFFs...
  Mafra: example 1 (point 0)
  Mafra: example 2 (point 1)
  Mafra: example 3 (point 2)
  Mafra: example 4 (point 3)
  Mafra: example 5 (point 4)
  Mafra: example 6 (point 5)
  Mafra: example 7 (point 6)
  Mafra: example 8 (point 7)
  Mafra: example 9 (point 8)
  Mafra: example 10 (point 9)
  Mafra: example 11 (point 10)
  Mafra: example 12 (point 11)
  Mafra: example 13 (point 12)
  Mafra: example 14 (point 13)
  Mafra: example 15 (point 14)
  Mafra: example 16 (point 15)
  Mafra: example 17 (point 16)
  Mafra: example 18 (point 17)
  Mafra: example 19 (point 18)
  Mafra: example 20 (point 19)
  Mafra: example 21 (point 20)
  Mafra: example 22 (point 21)
  Mafra: example 23 (point 22)
  Mafra: example 24 (point 23)
  Mafra: example 25 (point 24)
  Mafra: example 26 (point 25)
  Mafra: example 27 (point 26)
  Mafra: example 28 (point 27)
  Mafra: example 29 (point 28)
  Mafra: example 30 (point 29)
  Mafra: example 31 (point 30)
  Ma

In [24]:
print("Número de rasters no patch:", len(sample['patches']))
print("Nomes:", list(sample['patches'].keys()))

Número de rasters no patch: 306
Nomes: ['VPA_20200000_ALOS2_BETAHH_A', 'VPA_20200000_ALOS2_BETAHV_A', 'VPA_20200000_ALOS2_BETAND_A', 'VPA_20200000_ALOS2_BETARATIO_A', 'VPA_20200000_ALOS2_GAMMAHH_A', 'VPA_20200000_ALOS2_GAMMAHV_A', 'VPA_20200000_ALOS2_GAMMAND_A', 'VPA_20200000_ALOS2_GAMMARATIO_A', 'VPA_20200000_ALOS2_GLCMBETAHHASM_A', 'VPA_20200000_ALOS2_GLCMBETAHHCONTRAST_A', 'VPA_20200000_ALOS2_GLCMBETAHHCORRELATION_A', 'VPA_20200000_ALOS2_GLCMBETAHHDISSIMILARITY_A', 'VPA_20200000_ALOS2_GLCMBETAHHENERGY_A', 'VPA_20200000_ALOS2_GLCMBETAHHENTROPY_A', 'VPA_20200000_ALOS2_GLCMBETAHHHOMOGENEITY_A', 'VPA_20200000_ALOS2_GLCMBETAHHMAX_A', 'VPA_20200000_ALOS2_GLCMBETAHHMEAN_A', 'VPA_20200000_ALOS2_GLCMBETAHHVARIANCE_A', 'VPA_20200000_ALOS2_GLCMBETAHVASM_A', 'VPA_20200000_ALOS2_GLCMBETAHVCONTRAST_A', 'VPA_20200000_ALOS2_GLCMBETAHVCORRELATION_A', 'VPA_20200000_ALOS2_GLCMBETAHVDISSIMILARITY_A', 'VPA_20200000_ALOS2_GLCMBETAHVENERGY_A', 'VPA_20200000_ALOS2_GLCMBETAHVENTROPY_A', 'VPA_20200000_ALOS2_

In [25]:
# =====================================================================
# VALIDATION REPORT
# =====================================================================

print("\n" + "="*70)
print("MULTISOURCE PATCH EXTRACTION - VALIDATION REPORT")
print("="*70)

for area_name, result in all_results.items():
    
    X = result['X']
    y = result['y']
    samples = result['samples']
    raster_names = result['raster_names']
    
    print(f"\nArea: {area_name}")
    print("-" * 70)
    
    # Sample statistics
    total_samples = len(samples)
    valid_samples = sum(1 for s in samples if s['valid'])
    invalid_samples = total_samples - valid_samples
    
    print(f"Total reference points: {total_samples}")
    print(f"Valid patch samples: {valid_samples}")
    print(f"Invalid patch samples: {invalid_samples} ({100*invalid_samples/total_samples:.1f}%)")
    
    # Tensor information
    print(f"\nTensor information:")
    print(f"  Shape: {X.shape} (N, C, H, W)")
    print(f"  Size: {X.nbytes / 1e9:.3f} GB")
    print(f"  Data type: {X.dtype}")
    
    # Channel information
    print(f"\nChannels ({len(raster_names)} total):")
    for i, name in enumerate(raster_names[:10]):
        print(f"  {i}: {name}")
    if len(raster_names) > 10:
        print(f"  ... and {len(raster_names) - 10} more")
    
    # NaN statistics
    nan_count = np.isnan(X).sum()
    total_vals = X.size
    nan_pct = 100 * nan_count / total_vals
    
    print(f"\nNaN statistics:")
    print(f"  Total NaN values: {nan_count}/{total_vals} ({nan_pct:.2f}%)")
    
    # Interpolation method usage
    print(f"\nInterpolation methods used:")
    bicubic_total = sum(s['metadata'][list(s['metadata'].keys())[0]]['bicubic'] 
                       for s in samples if s['valid'] and s['metadata'])
    bilinear_total = sum(s['metadata'][list(s['metadata'].keys())[0]]['bilinear'] 
                        for s in samples if s['valid'] and s['metadata'])
    nearest_total = sum(s['metadata'][list(s['metadata'].keys())[0]]['nearest'] 
                       for s in samples if s['valid'] and s['metadata'])
    
    total_interp = bicubic_total + bilinear_total + nearest_total
    if total_interp > 0:
        print(f"  Bicubic: {100*bicubic_total/total_interp:.1f}%")
        print(f"  Bilinear: {100*bilinear_total/total_interp:.1f}%")
        print(f"  Nearest: {100*nearest_total/total_interp:.1f}%")
    
    # Target statistics
    print(f"\nTarget value (y) statistics:")
    print(f"  Mean: {np.nanmean(y):.6f}")
    print(f"  Std:  {np.nanstd(y):.6f}")
    print(f"  Min:  {np.nanmin(y):.6f}")
    print(f"  Max:  {np.nanmax(y):.6f}")
    print(f"  Median: {np.nanmedian(y):.6f}")
    
    # Output files
    area_output_dir = os.path.join(OUTPUT_BASE_DIR, area_name)
    print(f"\nOutput files:")
    print(f"  {os.path.join(area_output_dir, 'X.npy')}")
    print(f"  {os.path.join(area_output_dir, 'y.npy')}")
    print(f"  {os.path.join(area_output_dir, 'metadata.json')}")

print("\n" + "="*70)
print("REPORT COMPLETE - Ready for deep learning!")
print("="*70)


MULTISOURCE PATCH EXTRACTION - VALIDATION REPORT

Area: Mafra
----------------------------------------------------------------------
Total reference points: 463
Valid patch samples: 389
Invalid patch samples: 74 (16.0%)

Tensor information:
  Shape: (389, 306, 15, 15) (N, C, H, W)
  Size: 0.107 GB
  Data type: float32

Channels (306 total):
  0: Mafra_20200000_ALOS2_BETAHH_A
  1: Mafra_20200000_ALOS2_BETAHV_A
  2: Mafra_20200000_ALOS2_BETAND_A
  3: Mafra_20200000_ALOS2_BETARATIO_A
  4: Mafra_20200000_ALOS2_GAMMAHH_A
  5: Mafra_20200000_ALOS2_GAMMAHV_A
  6: Mafra_20200000_ALOS2_GAMMAND_A
  7: Mafra_20200000_ALOS2_GAMMARATIO_A
  8: Mafra_20200000_ALOS2_GLCMBETAHHASM_A
  9: Mafra_20200000_ALOS2_GLCMBETAHHCONTRAST_A
  ... and 296 more

NaN statistics:
  Total NaN values: 39725/26782650 (0.15%)

Interpolation methods used:
  Bicubic: 99.2%
  Bilinear: 0.8%
  Nearest: 0.0%

Target value (y) statistics:
  Mean: 0.000000
  Std:  0.000000
  Min:  0.000000
  Max:  0.000000
  Median: 0.000000

O

In [26]:
# =====================================================================
# LOAD AND VERIFY SAVED TENSORS
# =====================================================================

def load_area_tensors(area_name, output_base_dir=OUTPUT_BASE_DIR):
    """
    Load tensors and metadata for an area.
    """
    
    area_dir = os.path.join(output_base_dir, area_name)
    
    X = np.load(os.path.join(area_dir, 'X.npy'))
    y = np.load(os.path.join(area_dir, 'y.npy'))
    
    with open(os.path.join(area_dir, 'metadata.json'), 'r') as f:
        metadata = json.load(f)
    
    return X, y, metadata


# Example: load tensors for first area
if all_results:
    first_area = list(all_results.keys())[0]
    print(f"Loading tensors for {first_area}...")
    
    X, y, metadata = load_area_tensors(first_area)
    
    print(f"\nLoaded successfully!")
    print(f"  X shape: {X.shape}")
    print(f"  y shape: {y.shape}")
    print(f"  Metadata: {json.dumps(metadata, indent=2)}")

Loading tensors for Mafra...

Loaded successfully!
  X shape: (389, 306, 15, 15)
  y shape: (389,)
  Metadata: {
  "patch_size": 15,
  "pixel_size": 25,
  "num_channels": 306,
  "num_samples": 389,
  "raster_names": [
    "Mafra_20200000_ALOS2_BETAHH_A",
    "Mafra_20200000_ALOS2_BETAHV_A",
    "Mafra_20200000_ALOS2_BETAND_A",
    "Mafra_20200000_ALOS2_BETARATIO_A",
    "Mafra_20200000_ALOS2_GAMMAHH_A",
    "Mafra_20200000_ALOS2_GAMMAHV_A",
    "Mafra_20200000_ALOS2_GAMMAND_A",
    "Mafra_20200000_ALOS2_GAMMARATIO_A",
    "Mafra_20200000_ALOS2_GLCMBETAHHASM_A",
    "Mafra_20200000_ALOS2_GLCMBETAHHCONTRAST_A",
    "Mafra_20200000_ALOS2_GLCMBETAHHCORRELATION_A",
    "Mafra_20200000_ALOS2_GLCMBETAHHDISSIMILARITY_A",
    "Mafra_20200000_ALOS2_GLCMBETAHHENERGY_A",
    "Mafra_20200000_ALOS2_GLCMBETAHHENTROPY_A",
    "Mafra_20200000_ALOS2_GLCMBETAHHHOMOGENEITY_A",
    "Mafra_20200000_ALOS2_GLCMBETAHHMAX_A",
    "Mafra_20200000_ALOS2_GLCMBETAHHMEAN_A",
    "Mafra_20200000_ALOS2_GLCMBETAHHVARIA